In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Đọc dữ liệu thô
df = pd.read_csv('../data/raw/lung_cancer.csv')
print(f"Kích thước bộ dữ liệu ban đầu: {df.shape}")

Kích thước bộ dữ liệu ban đầu: (5000, 30)


In [2]:
# 2. Tách toàn bộ 29 features (X) và target (y)
X = df.drop(columns=['lung_cancer_risk'])
y = df['lung_cancer_risk']

# Lưu lại tên các cột để dùng sau này (vì sklearn sẽ biến DataFrame thành mảng Numpy)
feature_names = X.columns

# 3. Điền khuyết dữ liệu bằng giá trị Trung vị (Median Imputation)
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_names)

print("Đã kiểm tra và xử lý xong các giá trị thiếu (Missing Values)!")

Đã kiểm tra và xử lý xong các giá trị thiếu (Missing Values)!


In [3]:
# 4. Chia tập dữ liệu (80% Train - 20% Test)
# stratify=y đảm bảo tỉ lệ người mắc bệnh ở tập Train và Test là như nhau
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (4000, 29)
Kích thước tập Test: (1000, 29)


In [4]:
# 5. Xác định các biến liên tục cần chuẩn hóa
continuous_cols = [
    'age', 'education_years', 'income_level', 'smoking_years', 'cigarettes_per_day', 
    'pack_years', 'air_pollution_index', 'bmi', 'oxygen_saturation', 
    'fev1_x10', 'crp_level', 'exercise_hours_per_week', 'diet_quality', 
    'alcohol_units_per_week', 'healthcare_access'
]

scaler = StandardScaler()

# CHỈ FIT trên tập Train để tránh rò rỉ dữ liệu (Data Leakage)
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])

# Dùng bộ scaler đó transform cho tập Test
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Đã chuẩn hóa (Scale) xong các biến liên tục!")

Đã chuẩn hóa (Scale) xong các biến liên tục!


In [5]:
# 6. Gộp X và y lại và lưu thành file CSV mới
train_processed = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_processed = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

# Lưu vào thư mục processed
train_processed.to_csv('../data/processed/train_processed.csv', index=False)
test_processed.to_csv('../data/processed/test_processed.csv', index=False)

print("Đã xuất file thành công vào thư mục data/processed/!")
print("Sẵn sàng cho bước 03_Modeling_Nature.ipynb")

Đã xuất file thành công vào thư mục data/processed/!
Sẵn sàng cho bước 03_Modeling_Nature.ipynb


In [6]:
import joblib
import os

# Tạo thư mục models nếu chưa có
os.makedirs('../models', exist_ok=True)

# Lưu bộ chuẩn hóa (scaler) ngay tại nơi nó được tạo ra
joblib.dump(scaler, '../models/scaler.pkl')
print("Đã lưu file scaler.pkl thành công!")

Đã lưu file scaler.pkl thành công!
